# `dippa.profiles` walkthrough

This notebook does two jobs:

1. **A manual sanity check while developing** — rerun this after changing
   anything in `profiles.py` and eyeball the plots; catches obviously wrong
   changes faster than digging through `pytest -v` output.
2. **A first example for users**, once the API stabilises — this is the
   easiest way to see what the peak-shape functions actually produce.

Status: pre-alpha. This only covers the profile/peak-shape functions —
there is no fitter yet (see `CLAUDE.md` for what's next). Nothing here
should be used for real analysis.

## 1. A single symmetric peak

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dippa.profiles import pseudo_voigt, asymmetric_pseudo_voigt, evaluate_peak, evaluate_pattern

x = np.linspace(-0.05, 0.05, 2000)

fig, ax = plt.subplots(figsize=(6, 4))
for eta, label in [(0.0, "eta=0 (pure Gaussian)"), (0.5, "eta=0.5"), (1.0, "eta=1 (pure Lorentzian)")]:
    y = pseudo_voigt(x, x0=0.0, amplitude=1.0, fwhm=0.02, eta=eta)
    ax.plot(x, y, label=label)
ax.set_xlabel("x - x0")
ax.set_ylabel("intensity")
ax.set_title("pseudo_voigt: same FWHM, varying Lorentzian fraction")
ax.legend()
plt.show()

Note the Lorentzian curve has a sharper centre and much heavier tails than
the Gaussian for the *same* nominal FWHM — this is why a single width
number is never quite enough to describe a real diffraction peak, and why
`eta` exists as a separate fitted parameter.

## 2. Asymmetric peak — different shape either side of the centre

In [ ]:
x = np.linspace(-0.05, 0.05, 2000)

# Deliberately very different left/right FWHM and eta to make the asymmetry obvious.
y_asym = asymmetric_pseudo_voigt(
    x, x0=0.0, amplitude=1.0,
    fwhm_right=0.008, eta_left=0.2,
    fwhm_left=0.02, eta_right=0.8,
)
y_sym_right = pseudo_voigt(x, 0.0, 1.0, 0.008, 0.8)  # what the right side alone would look like, symmetric

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, y_asym, label="asymmetric_pseudo_voigt")
ax.plot(x, y_sym_right, "--", label="pseudo_voigt using only the right-side params", alpha=0.6)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set_xlabel("x - x0")
ax.set_title("Left side uses (fwhm_left, eta_left); right side uses (fwhm_right, eta_right)")
ax.legend()
plt.show()

**Parameter order gotcha, worth seeing directly rather than just reading about it:**
the 6-parameter convention is
`[x0, amplitude, fwhm_right, eta_left, fwhm_left, eta_right]` — note that
`eta_left` (index 3) and `fwhm_left` (index 4) are *not* adjacent to each
other in a way that groups by side. This is confirmed correct against the
original tool's real output (see `AUDIT.md` §3, §9), not a design choice
made here — don't "fix" the ordering without re-running the parity test in
`tests/test_profiles.py`.

## 3. The Kα1/Kα2 doublet

In [ ]:
x = np.linspace(0.45, 0.51, 4000)
params = np.array([0.478, 1.0, 0.0015, 0.5, 0.0015, 0.5])  # x0, amplitude, fwhm_r, eta_l, fwhm_l, eta_r

single = evaluate_peak(x, params, tube=None)
doublet = evaluate_peak(x, params, tube="Co")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, single, label="tube=None (single peak)")
ax.plot(x, doublet, label="tube='Co' (Ka1 + Ka2 doublet)")
ax.set_xlabel("g (1/d, A^-1)")
ax.legend()
ax.set_title("The Ka2 component sits to the right of x0 at half the intensity")
plt.show()

**Don't assume a dataset's saved preferences tell you whether the doublet was applied.**
See the next section — for the bundled reference dataset, the settings file
says "fit the doublet" and it's simply wrong for that saved fit.

## 4. Real parity check: reproducing the original tool's saved fit

In [ ]:
from pathlib import Path
import scipy.io

fixtures = Path("../tests/fixtures")
fit = scipy.io.loadmat(fixtures / "reference_fit.mat")
data = scipy.io.loadmat(fixtures / "reference_data.mat")

aa = fit["aa"]
d = data["data"]
x_real, y_real = d[:, 0], d[:, 1]

print(f"{aa.shape[1] - 1} peaks, {len(x_real)} measured points")

In [ ]:
model_no_doublet = evaluate_pattern(x_real, aa, tube=None)
model_with_doublet = evaluate_pattern(x_real, aa, tube="Co")

def r_squared(y, model):
    ss_res = np.sum((y - model) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot

print(f"R^2 without doublet: {r_squared(y_real, model_no_doublet):.4f}")
print(f"R^2 with doublet:    {r_squared(y_real, model_with_doublet):.4f}")

The `genset.mat` bundled alongside this example has `alpha2 = 0`, which in
the *original* MATLAB tool's inverted convention means "fit the doublet" —
and yet the doublet-off model is the one that actually reproduces the real
pattern. See `AUDIT.md` §9 for the full story. Plotting both makes it
obvious why:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, model, title in [
    (axes[0], model_no_doublet, "No doublet (correct for this fit)"),
    (axes[1], model_with_doublet, "With Co doublet (wrong for this fit)"),
]:
    ax.plot(x_real, y_real, ".", markersize=1, alpha=0.4, label="measured")
    ax.plot(x_real, model, linewidth=1, label="ported model")
    ax.set_xlim(0.47, 0.49)  # zoom on the first peak, where the difference is clearest
    ax.set_xlabel("g (1/d, A^-1)")
    ax.set_title(title)
    ax.legend()

axes[0].set_ylabel("intensity")
plt.tight_layout()
plt.show()

The doublet-applied model visibly over-predicts intensity to the right of
the peak centre — the spurious second component. The no-doublet model
tracks the real (noisy) data closely across the whole pattern, not just
this one peak.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_real, y_real, ".", markersize=1, alpha=0.3, label="measured")
ax.plot(x_real, model_no_doublet, linewidth=1, label="ported model (no doublet)")
ax.set_xlabel("g (1/d, A^-1)")
ax.set_ylabel("intensity")
ax.set_title("Full pattern, all 10 peaks")
ax.legend()
plt.show()